<a href="https://www.su.se/forskning/forskargrupper/data-science-research-group"><img src="https://www.pngkey.com/png/detail/853-8536888_stockholm-university-stockholm-university-logo.png" width="400" align="center"></a>

<h1><center>Homework Lab 2</center></h1>

# Dimensionality Reduction and Clustering

## Objective:

- Generate two datasets: one spherical and one non-spherical.

- Implement KMeans++ and Agglomerative Hierarchical Clustering from scratch.

- Reduce dimensions to 2D for visualization.

- Evaluate clustering performance using silhouette coefficient and purity index.

- Compare the performance of both algorithms and reach suitable conclusions.

## General Guidelines

- Fill up the missing codes (TODO) using the hints given.
- You may also use the lab exercises done in class to get some ideas
- #### PLEASE DO NOT CHANGE THE NAMES OF FUNCTIONS AND PRESPECIFIED VARIABLES (for automated testing purpose).

## Task 1: Generate the dataset

In [1]:
## Task 1: Generate the dataset

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.decomposition import PCA
from collections import defaultdict
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mode

def generate_datasets():
    """Generates two datasets: one spherical and one non-spherical."""
    # TODO: Fill the missing code
    # Hint: Use make_blobs() method imported from sklearn.datasets
    # Both datasets should have 300 samples
    X_spherical, y_spherical = make_blobs(n_samples=300, centers=3, random_state=42) # There should be 3 centers, random_state=42
    X_nonspherical, y_nonspherical = make_moons(n_samples=300, noise=0.05, random_state=42) # noise=0.05, random_state=42
    return (X_spherical, y_spherical), (X_nonspherical, y_nonspherical)

# Load datasets
spherical_data, spherical_labels = generate_datasets()[0]
nonspherical_data, nonspherical_labels = generate_datasets()[1]

In [ ]:
# This part of code (this cell) is just for your better understanding. DO NOT PASTE IT TO YOUR FINAL SUBMISSIONS.

# Write a function to plot a dataset
def plot_dataset(X, y, title):
    """Plots dataset points with labels."""
    
# Plot datasets
plot_dataset(____) ## Fill in the blank spaces with appropriate arguments to plot
plot_dataset(____) ## Fill in the blank spaces with appropriate arguments to plot

# Task 2: Kmeans++ Clustering from scratch
## Fill up the missing codes to complete the functions for K-Means++ Clustering from scratch. 

In [ ]:
# TODO: Implement Kmeans++ algorithm from scratch
# Hint: You can take help from the kmeans++ algorithm implemented in the file "Clustering_Lab_With_Answers.ipynb". 
# The file is uploaded in nextiLearn. The code is a slightly different organization of that done in class. But the basic ideas are same.
# PLEASE DO NOT CHANGE THE NAMES OF FUNCTIONS AND PRESPECIFIED VARIABLES (for automated testing purpose).

def initialize_centroids(X, k):
    """Initializes centroids using the KMeans++ strategy."""
    np.random.seed(42)
    centroids = []
    first_centroid = X[np.random.choice(X.shape[0])] # Fillup the blank space
    centroids.append(first_centroid)
    
    '''Write your code here'''
    for i in range(1, k):
        distances = np.min([np.linalg.norm(X - c, axis=1) for c in centroids], axis=1)
        weights = distances ** 2
        next_centroid_index = np.random.choice(X.shape[0], p=weights / np.sum(weights))
        next_centroid = X[next_centroid_index]
        centroids.append(next_centroid)
    return np.array(centroids)

def assign_clusters(X, centroids):
    """Assigns each point to the nearest centroid."""
    distances = []
    for c in centroids:
        dist = np.linalg.norm(X - c, axis=1) # Calculate distance
        distances.append(dist)
    distances_array = np.array(distances).T
    labels = np.argmin(distances_array, axis=1)
    # TODO: Write the code to get the updated labels
    # Hint: convert distances to an np array, and the use argmin() function to get the labels
    return labels

def update_centroids(X, labels, k):
    """Recomputes centroids as the mean of assigned points."""
    new_centroids = []
    # TODO: Write your code here
    for i in range(k):
        points_in_cluster = X[labels == i]
        if points_in_cluster.shape[0] > 0:
            new_centroid = np.mean(points_in_cluster, axis=0)
            new_centroids.append(new_centroid)
        else:
            new_centroid = X[np.random.choice(X.shape[0])]
            new_centroids.append(new_centroid)
    '''Hints: Iterate k times, and for each label (cluster), find the centroid of the cluster using mean() function. 
    Add the new centroid to the list new_centroids'''
    return np.array(new_centroids)

def kmeans(X, k, max_iters=100):
    """Performs KMeans clustering."""
    centroids = initialize_centroids(X, k) # TODO: Fill in the missing arguments
    
    # TODO: Write your code here
    for i in range(max_iters):
        labels = assign_clusters(X, centroids)
        new_centroids = update_centroids(X, labels, k)
        if np.all(centroids == new_centroids):
            break
        centroids = new_centroids
    return labels

In [ ]:
# This part of code (this cell) is just for your better understanding. DO NOT PASTE IT TO YOUR FINAL SUBMISSIONS.

kmeans_labels_spherical = kmeans(spherical_data, k=3)
kmeans_labels_nonspherical = kmeans(nonspherical_data, k=2)

#Plot the clustering results
plot_dataset(spherical_data, kmeans_labels_spherical, "KMeans on Spherical Data")
plot_dataset(nonspherical_data, kmeans_labels_nonspherical, "KMeans on Non-Spherical Data")

# Task 3: Agglomerative Hierarchical Clustering from scratch
## Fill up the missing codes to complete the functions for Agglomerative Hierarchical Clustering from scratch. 

In [ ]:
## Task 3
# TODO: Fill up the missing codes, blank spaces
from scipy.spatial.distance import pdist, squareform

def compute_distance_matrix(X):
    """Computes the pairwise distance matrix."""
    pairwise_distances = pdist(X)  # TODO, Hint: Compute all pairwise distances using pdist() function
    distance_matrix = squareform(pairwise_distances)  # TODO, Hint: Convert to square matrix format
    return distance_matrix

def find_closest_clusters(distances):
    """Finds the indices of the two closest clusters."""
    min_dist_index = np.argmin(distances)  # TODO, Hint: Get index of the smallest value using argmin() function
    i, j = np.unravel_index(min_dist_index, distances.shape)
    return (i,j) if i < j else (j, i)# TODO, Hint: Convert to row, col indices using np.unravel() function with appropriate arguments
    return i, j

def update_distances(distances, clusters, i, j):
    """Updates the distance matrix after merging clusters i and j."""
    for idx in clusters:
        if idx != i:  # Skip the merged cluster
            cluster_distances = []  # Create an empty list to store distances
            for point_i in clusters[i]:
                min_distance = float('inf')
                if clusters[j]:
                    for point_j in clusters[j]:
                        dist = np.linalg.norm(point_i - point_j) 
                        if dist < min_distance:
                            min_distance = dist
                    cluster_distances.append(min_distance)
                else:
                    min_distance = float('inf')
                    cluster_distances.append(min_distance)   
            distances[idx, i] = distances[i, idx] = min(cluster_distances)
            # TODO, Hint: Calculate the distance between points
            # TODO: Loop through each point in cluster i
            # TODO: Append the corresponding distance
            # TODO: Update "distances" using Minimum linkage
    distances[:, j] = distances[j, :] = np.inf  # Set merged cluster to infinity to ignore it


def agglomerative_clustering(X, k):
    """Performs agglomerative hierarchical clustering using minimum distance linkage."""
    num_points = len(X)
    
    # Initialize each point as its own cluster
    clusters = {}  # Create an empty dictionary
    for i in range(num_points):  # Loop through each point index
        clusters[i] = [i]  # Assign each point to its own cluster
    
    # TODO: Compute distances matrix using the function compute_distance_matrix() defined earlier
    # TODO: Set diagonal to infinity to ignore self-distances
    distances = compute_distance_matrix(X)
    np.fill_diagonal(distances, np.inf)
    
    while len(clusters) > k:
        i, j = find_closest_clusters(distances)# Get the closest clusters using the function find_closest_clusters(distances)
        
        # Merge cluster j into cluster i
        clusters[i].extend(clusters[j])
        del clusters[j]
        update_distances(distances, clusters, i, j)
        # TODO: Update the distance matrix using the function update_distances() defined earlier
    
    # Assign labels to points based on final clusters
    labels = np.zeros(num_points)
    for cluster_id, points in enumerate(clusters.values()):
        for point in points:
            labels[point] = cluster_id
    # TODO: Complete the code here to get the labels. Use hints below:
    ''' enumerate over clusters values, and for each point, assign the corresponding labor as equal to the cluster id'''
    
    return labels

In [ ]:
# This part of code (this cell) is just for your better understanding. DO NOT PASTE IT TO YOUR FINAL SUBMISSIONS.
agglomerative_labels_spherical = agglomerative_clustering(spherical_data, k=3)
agglomerative_labels_nonspherical = agglomerative_clustering(nonspherical_data, k=2)

# Plot the clustering results
plot_dataset(spherical_data, agglomerative_labels_spherical, "Agglomerative Clustering on Spherical Data")
plot_dataset(nonspherical_data, agglomerative_labels_nonspherical, "Agglomerative Clustering on Non-Spherical Data")

# Task 4: Performance Evaluation
## Fill up the missing codes

In [ ]:
# Task 4: Performance Evaluation
#Internal Evaluation using Silhouette score

# Fill up the missing codes to implement Silhoutte score evaluation from scratch
from sklearn.metrics import pairwise_distances

def silhouette_score(X, labels):
    """Computes silhouette score for clustering."""
    distances = pairwise_distances(X)#TODO, Hint: Find pairwise distances in X using the function pairwise_distances(...)
    unique_labels = np.unique(labels)  # From the labels, find the unique labels using the function np.unique(...)
    
    # Compute intra-cluster distance (a)
    a = []
    for i in range(len(X)):
        same_cluster = labels == labels[i]  # Mask for same cluster points
        intra_distances = distances[i][same_cluster] # TODO, Hint: Get distances to same cluster points
        avg_distance = np.mean(intra_distances) if intra_distances.size > 0 else 0  # TODO, Hint:Compute mean value of intra_distances
        a.append(avg_distance) 
    a = np.array(a)
    
    # Compute nearest-cluster distance (b)
    b = []
    for i in range(len(X)):
        other_cluster_distances = []  # Store avg distances to other clusters
        for l in unique_labels:
            if l == labels[i]:
                continue
            other_cluster = labels == l# TODO, Hint: Write a condition to skip same clusters
                # TODO, Hint: Mask for other cluster (like in previous loop)
            cluster_distances = distances[i][other_cluster] # TODO, Hint: Extract distances with other cluster mask
            if cluster_distances.size > 0:  # Ensure it's not empty
                avg_distance = np.mean(cluster_distances) # TODO, Hint: Compute mean distance
                other_cluster_distances.append(avg_distance) 
        # Handle the case where other_cluster_distances is empty
        if other_cluster_distances:
            min_distance = min(other_cluster_distances) if other_cluster_distances 
        else :
            min_distance = 0# TODO, Hint: Compute minimum of other_cluster_distances if the list is not empty
            # Default value when no other clusters exist
        b.append(min_distance)  # Take the minimum avg distance
    b = np.array(b)
    
    return np.mean((b - a) / np.maximum(a, b))

#External Evaluation using purity score
from sklearn import metrics
def purity_score(y_true, y_pred):
    contingency_matrix = metrics.confusion_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)
    # TODO: Write the code to calculate purity score
    # Hint: Use the exercise in lab class.
    # return purity


# Task 5: Run the Clustering algorithms, and print the evaluation results.

In [ ]:
## Task 5
# Run the clustering algorithms
# TODO: Run both the clustering algortihms for both spherical and non-sperical data (so in total 4 function calls)

# Kmeans++
kmeans_sph = kmeans(spherical_data, k=3)
kmeans_sph_labels = kmeans_sph.fit_predict(spherical_data)
kmeans_non = kmeans(nonspherical_data, k=2)
kmeans_non_labels = kmeans_non.fit_predict(nonspherical_data)

# Agglomerative
agglo_sph_labels = agglomerative_clustering(spherical_data, 3)
agglo_non_labels = agglomerative_clustering(nonspherical_data, 2)
# Compute and print the evaluation metrics
s1 = silhouette_score(spherical_data, kmeans_sph_labels)
s2 = silhouette_score(spherical_data, agglo_sph_labels)
s3 = silhouette_score(nonspherical_data, kmeans_non_labels)
s4 = silhouette_score(nonspherical_data, agglo_non_labels)

p1 = purity_score(spherical_labels, kmeans_sph_labels)
p2 = purity_score(spherical_labels, agglo_sph_labels)
p3 = purity_score(nonspherical_labels, kmeans_non_labels)
p4 = purity_score(nonspherical_labels, agglo_non_labels)
# TODO: Compute both the silhoutte scores and purity scores - each for both kmeans++ algorithm and aglomerative algorithm, for both datasets.
# So in total, 8 function calls. 
# Print all the 8 values.
print("Silhouette Scores:")
print(f"KMeans++ (Spherical):     {s1:.4f}")
print(f"Agglomerative (Spherical): {s2:.4f}")
print(f"KMeans++ (Non-Spherical): {s3:.4f}")
print(f"Agglomerative (Non-Spherical): {s4:.4f}")

print("\nPurity Scores:")
print(f"KMeans++ (Spherical):     {p1:.4f}")
print(f"Agglomerative (Spherical): {p2:.4f}")
print(f"KMeans++ (Non-Spherical): {p3:.4f}")
print(f"Agglomerative (Non-Spherical): {p4:.4f}")

# Food for thought and further scope of learning (optional, but recommended)
- Which algorithms work (and not work) for the different datasets? Why?
- Try implementing DBSCAN clustering algorithm and compare the performances
- Try the same exercises by taking a higher dimensional data, performing dimensionality reduction, running clustering algorithms, and comparing performances.